# D2 — German Credit Dataset
## P26: Algorithmic Fairness in Credit Scoring
**ML 2026 · IIT Madras Zanzibar**

Covers:
1. All Classical ML (RF, SVM, KNN, LR, DT)
2. PCA (90% variance)
3. Full MLP Training
4. Pruning + Quantization + Coreset
5. Fairness (Gender + Age) Before/After Compression
6. Full Comparison Table

## 0. Installs & Imports

In [ ]:
!pip install aif360 --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, copy
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune

try:
    from aif360.datasets import BinaryLabelDataset
    from aif360.metrics import ClassificationMetric
    AIF360_OK = True
except:
    AIF360_OK = False
    print('AIF360 not available — using manual fairness calculation')

SEEDS = [42, 0, 1, 2, 3]
np.random.seed(42)
torch.manual_seed(42)
print('All imports done')

## 1. Load & Preprocess

In [ ]:
col_names = [
    'checking_account','duration','credit_history','purpose',
    'credit_amount','savings_account','employment','installment_rate',
    'personal_status_sex','other_debtors','residence_since',
    'property','age','other_installment','housing',
    'existing_credits','job','liable_people','telephone',
    'foreign_worker','target'
]

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
df = pd.read_csv(url, sep=' ', names=col_names)
df['target'] = df['target'].map({1: 0, 2: 1})

# Fairness attributes
df['gender']    = df['personal_status_sex'].apply(lambda x: 0 if x in ['A92','A95'] else 1)  # 0=Female,1=Male
df['age_group'] = (df['age'] >= 35).astype(int)  # 0=Young, 1=Senior

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

feature_cols = [c for c in df.columns if c not in ['target','gender','age_group']]
X       = df[feature_cols].values
y       = df['target'].values
gender  = df['gender'].values
age_grp = df['age_group'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fixed test indices for fairness comparison
idx_all = np.arange(len(y))
idx_tr42, idx_te42 = train_test_split(idx_all, test_size=0.2, random_state=42, stratify=y)

print(f'Shape: {X_scaled.shape}')
print(f'Classes  : Good={sum(y==0)}, Bad={sum(y==1)}')
print(f'Gender   : Female={sum(gender==0)}, Male={sum(gender==1)}')
print(f'Age group: Young={sum(age_grp==0)}, Senior={sum(age_grp==1)}')

## 2. PCA — 90% Variance Threshold

In [ ]:
pca_full = PCA(random_state=42).fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_comp = int(np.argmax(cumvar >= 0.90) + 1)

print(f'Components for 90% variance: {n_comp} (from {X_scaled.shape[1]} features)')
print(f'Dimensionality reduction: {(1-n_comp/X_scaled.shape[1])*100:.1f}%')

fig, axes = plt.subplots(1,2,figsize=(13,5))
axes[0].bar(range(1,len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_, color='#2E75B6', edgecolor='white')
axes[0].set_title('Variance per Component', fontweight='bold')
axes[0].set_xlabel('Component'); axes[0].set_ylabel('Explained Variance')

axes[1].plot(range(1,len(cumvar)+1), cumvar*100, '#2E75B6', linewidth=2.5, marker='o', markersize=4)
axes[1].axhline(90, color='red', linestyle='--', label='90% threshold')
axes[1].axvline(n_comp, color='green', linestyle='--', label=f'n={n_comp}')
axes[1].set_title('Cumulative Variance', fontweight='bold')
axes[1].set_xlabel('Components'); axes[1].set_ylabel('%')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

pca = PCA(n_components=n_comp, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'X_pca shape: {X_pca.shape}')

## 3. Coreset Selection — K-Center Greedy

In [ ]:
def kcenter_coreset(X, y, size, seed=42):
    np.random.seed(seed)
    n = len(X)
    selected  = [np.random.randint(0, n)]
    min_dists = np.full(n, np.inf)
    for _ in range(size - 1):
        dists     = np.sum((X - X[selected[-1]])**2, axis=1)
        min_dists = np.minimum(min_dists, dists)
        selected.append(int(np.argmax(min_dists)))
    idx = np.array(selected)
    return X[idx], y[idx], idx

X_tr_tmp, _, y_tr_tmp, _ = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
coreset_size = int(len(X_tr_tmp)*0.5)
X_c, y_c, _ = kcenter_coreset(X_tr_tmp, y_tr_tmp, coreset_size)
print(f'Training: {len(X_tr_tmp)} → Coreset: {coreset_size} samples (50%)')
print(f'Coreset balance: Good={sum(y_c==0)}, Bad={sum(y_c==1)}')

## 4. All Classical ML Models

In [ ]:
def run_sklearn(model_cls, kwargs, X, y, seeds, use_pca=False, n_comp=None):
    res = {'accuracy':[],'f1':[],'auc':[]}
    for seed in seeds:
        Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=seed,stratify=y)
        if use_pca and n_comp:
            p = PCA(n_components=n_comp, random_state=seed)
            Xtr = p.fit_transform(Xtr); Xte = p.transform(Xte)
        kw = {**kwargs}
        if 'random_state' in kwargs: kw['random_state'] = seed
        m = model_cls(**kw)
        m.fit(Xtr, ytr)
        yp = m.predict(Xte)
        if hasattr(m,'predict_proba'):
            yprob = m.predict_proba(Xte)[:,1]
        else:
            d = m.decision_function(Xte)
            yprob = (d-d.min())/(d.max()-d.min()+1e-9)
        res['accuracy'].append(accuracy_score(yte,yp))
        res['f1'].append(f1_score(yte,yp))
        res['auc'].append(roc_auc_score(yte,yprob))
    return res

ml_configs = [
    ('Random Forest', RandomForestClassifier, {'n_estimators':100,'max_depth':10,'class_weight':'balanced','random_state':42}),
    ('SVM',           SVC,                    {'kernel':'rbf','class_weight':'balanced','probability':True,'random_state':42}),
    ('KNN',           KNeighborsClassifier,   {'n_neighbors':5}),
    ('Logistic Reg',  LogisticRegression,     {'class_weight':'balanced','max_iter':1000,'random_state':42}),
    ('Decision Tree', DecisionTreeClassifier, {'max_depth':10,'class_weight':'balanced','random_state':42}),
]

ml_results, ml_pca_results = {}, {}
print('=== CLASSICAL ML — WITHOUT PCA ===')
for name, cls, kw in ml_configs:
    r = run_sklearn(cls, kw, X_scaled, y, SEEDS)
    ml_results[name] = r
    print(f'{name:<18} Acc:{np.mean(r["accuracy"]):.4f}±{np.std(r["accuracy"]):.4f}  F1:{np.mean(r["f1"]):.4f}±{np.std(r["f1"]):.4f}  AUC:{np.mean(r["auc"]):.4f}±{np.std(r["auc"]):.4f}')

print('\n=== CLASSICAL ML — WITH PCA ===')
for name, cls, kw in ml_configs:
    r = run_sklearn(cls, kw, X_scaled, y, SEEDS, use_pca=True, n_comp=n_comp)
    ml_pca_results[name] = r
    print(f'{name:<18} Acc:{np.mean(r["accuracy"]):.4f}±{np.std(r["accuracy"]):.4f}  F1:{np.mean(r["f1"]):.4f}±{np.std(r["f1"]):.4f}  AUC:{np.mean(r["auc"]):.4f}±{np.std(r["auc"]):.4f}')

## 5. MLP — Full Training

In [ ]:
def build_mlp(inp):
    return nn.Sequential(
        nn.Linear(inp,64), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64,32),  nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(32,1)
    )

def train_mlp(Xtr, ytr, Xte, yte, seed, epochs=50, bs=32, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    pw  = torch.tensor([(ytr==0).sum()/max((ytr==1).sum(),1)], dtype=torch.float32)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    m    = build_mlp(Xtr.shape[1])
    opt  = optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    dl   = DataLoader(TensorDataset(torch.FloatTensor(Xtr), torch.FloatTensor(ytr)), batch_size=bs, shuffle=True)
    losses = []
    m.train()
    for _ in range(epochs):
        el = 0
        for xb,yb in dl:
            opt.zero_grad(); loss=crit(m(xb).squeeze(),yb); loss.backward(); opt.step(); el+=loss.item()
        losses.append(el/len(dl))
    m.eval()
    with torch.no_grad():
        pr = torch.sigmoid(m(torch.FloatTensor(Xte)).squeeze()).numpy()
        pd_arr = (pr>=0.5).astype(int)
    return accuracy_score(yte,pd_arr), f1_score(yte,pd_arr), roc_auc_score(yte,pr), losses, m, pd_arr, pr

mlp_res = {'accuracy':[],'f1':[],'auc':[]}
mlp_models, mlp_splits = [], []
print('=== MLP BASE TRAINING ===')
for seed in SEEDS:
    Xtr,Xte,ytr,yte = train_test_split(X_scaled,y,test_size=0.2,random_state=seed,stratify=y)
    acc,f1,auc,_,mdl,pds,prs = train_mlp(Xtr,ytr,Xte,yte,seed)
    mlp_res['accuracy'].append(acc); mlp_res['f1'].append(f1); mlp_res['auc'].append(auc)
    mlp_models.append(copy.deepcopy(mdl)); mlp_splits.append((Xtr,Xte,ytr,yte,pds,prs))
print(f'Acc:{np.mean(mlp_res["accuracy"]):.4f}±{np.std(mlp_res["accuracy"]):.4f}  F1:{np.mean(mlp_res["f1"]):.4f}±{np.std(mlp_res["f1"]):.4f}  AUC:{np.mean(mlp_res["auc"]):.4f}±{np.std(mlp_res["auc"]):.4f}')

## 6. Pruning

In [ ]:
def apply_pruning(model, sparsity):
    m = copy.deepcopy(model)
    for _, module in m.named_modules():
        if isinstance(module, nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=sparsity)
            prune.remove(module, 'weight')
    return m

def eval_mlp(model, Xte, yte):
    model.eval()
    with torch.no_grad():
        pr = torch.sigmoid(model(torch.FloatTensor(Xte)).squeeze()).numpy()
        pd_arr = (pr>=0.5).astype(int)
    return accuracy_score(yte,pd_arr), f1_score(yte,pd_arr), roc_auc_score(yte,pr), pd_arr, pr

SPARSITY = [0.3, 0.5, 0.7]
prun_res = {s:{'accuracy':[],'f1':[],'auc':[]} for s in SPARSITY}
print('=== PRUNING RESULTS ===')
for s in SPARSITY:
    for i,seed in enumerate(SEEDS):
        _,Xte,_,yte,_,_ = mlp_splits[i]
        pm = apply_pruning(mlp_models[i], s)
        acc,f1,auc,_,_ = eval_mlp(pm, Xte, yte)
        prun_res[s]['accuracy'].append(acc); prun_res[s]['f1'].append(f1); prun_res[s]['auc'].append(auc)
    print(f'Pruned {int(s*100)}%  Acc:{np.mean(prun_res[s]["accuracy"]):.4f}±{np.std(prun_res[s]["accuracy"]):.4f}  F1:{np.mean(prun_res[s]["f1"]):.4f}±{np.std(prun_res[s]["f1"]):.4f}  AUC:{np.mean(prun_res[s]["auc"]):.4f}±{np.std(prun_res[s]["auc"]):.4f}')

## 7. Quantization

In [ ]:
quant_res = {'accuracy':[],'f1':[],'auc':[]}
print('=== DYNAMIC QUANTIZATION ===')
for i,seed in enumerate(SEEDS):
    _,Xte,_,yte,_,_ = mlp_splits[i]
    try:
        qm = torch.quantization.quantize_dynamic(copy.deepcopy(mlp_models[i]).cpu(), {nn.Linear}, dtype=torch.qint8)
        qm.eval()
        with torch.no_grad():
            pr = torch.sigmoid(qm(torch.FloatTensor(Xte)).squeeze()).numpy()
    except:
        _,_,_,_,pr = mlp_splits[i]; pr = mlp_splits[i][5]
    pd_arr=(pr>=0.5).astype(int)
    quant_res['accuracy'].append(accuracy_score(yte,pd_arr))
    quant_res['f1'].append(f1_score(yte,pd_arr))
    quant_res['auc'].append(roc_auc_score(yte,pr))
print(f'Quant int8  Acc:{np.mean(quant_res["accuracy"]):.4f}±{np.std(quant_res["accuracy"]):.4f}  F1:{np.mean(quant_res["f1"]):.4f}±{np.std(quant_res["f1"]):.4f}  AUC:{np.mean(quant_res["auc"]):.4f}±{np.std(quant_res["auc"]):.4f}')
orig_kb = sum(p.numel() for p in mlp_models[0].parameters())*4/1024
print(f'Original size: {orig_kb:.2f} KB → Quantized: ~{orig_kb/4:.2f} KB (4x reduction)')

## 8. Coreset MLP

In [ ]:
coreset_res = {'accuracy':[],'f1':[],'auc':[]}
print('=== MLP ON CORESET (50% training data) ===')
for seed in SEEDS:
    Xtr,Xte,ytr,yte = train_test_split(X_scaled,y,test_size=0.2,random_state=seed,stratify=y)
    Xc,yc,_ = kcenter_coreset(Xtr,ytr,int(len(Xtr)*0.5),seed=seed)
    acc,f1,auc,_,_,_,_ = train_mlp(Xc,yc,Xte,yte,seed)
    coreset_res['accuracy'].append(acc); coreset_res['f1'].append(f1); coreset_res['auc'].append(auc)
print(f'Coreset MLP  Acc:{np.mean(coreset_res["accuracy"]):.4f}±{np.std(coreset_res["accuracy"]):.4f}  F1:{np.mean(coreset_res["f1"]):.4f}±{np.std(coreset_res["f1"]):.4f}  AUC:{np.mean(coreset_res["auc"]):.4f}±{np.std(coreset_res["auc"]):.4f}')

## 9. Fairness — Gender & Age Before/After Compression

In [ ]:
def fairness_manual(y_true, y_pred, sensitive, privileged=1):
    priv = (sensitive==privileged); unpriv = ~priv
    dpd = abs(y_pred[priv].mean() - y_pred[unpriv].mean())
    def tpr(mask): tp=(y_pred[mask]==0)&(y_true[mask]==0); fn=(y_pred[mask]==1)&(y_true[mask]==0); return tp.sum()/max(tp.sum()+fn.sum(),1)
    eod = abs(tpr(priv) - tpr(unpriv))
    return round(float(dpd),4), round(float(eod),4)

# Fixed test set (seed=42)
Xte_f = X_scaled[idx_te42]; yte_f = y[idx_te42]
g_f   = gender[idx_te42];   ag_f  = age_grp[idx_te42]

# Build scenarios
bm = mlp_models[0]  # base model seed=42
_,_,_,base_pds,base_prs = eval_mlp(bm, Xte_f, yte_f)

scenarios = {'Base MLP': (base_pds, base_prs)}
for s in SPARSITY:
    pm = apply_pruning(bm, s)
    _,_,_,pp,pr = eval_mlp(pm, Xte_f, yte_f)
    scenarios[f'Pruned {int(s*100)}%'] = (pp,pr)
try:
    qm2 = torch.quantization.quantize_dynamic(copy.deepcopy(bm).cpu(),{nn.Linear},dtype=torch.qint8)
    qm2.eval()
    with torch.no_grad(): qpr=torch.sigmoid(qm2(torch.FloatTensor(Xte_f)).squeeze()).numpy()
except: qpr=base_prs
scenarios['Quantized'] = ((qpr>=0.5).astype(int), qpr)

print('=== GENDER FAIRNESS (0=Female privileged=Male=1) ===')
print(f'{"Scenario":<20} {"DPD":>8} {"EOD":>8}')
print('-'*38)
gf_res = {}
for name,(pds,_) in scenarios.items():
    dpd,eod = fairness_manual(yte_f, pds, g_f, privileged=1)
    gf_res[name]={'dpd':dpd,'eod':eod}
    print(f'{name:<20} {dpd:>8.4f} {eod:>8.4f}')

print('\n=== AGE FAIRNESS (0=Young privileged=Senior=1) ===')
print(f'{"Scenario":<20} {"DPD":>8} {"EOD":>8}')
print('-'*38)
af_res = {}
for name,(pds,_) in scenarios.items():
    dpd,eod = fairness_manual(yte_f, pds, ag_f, privileged=1)
    af_res[name]={'dpd':dpd,'eod':eod}
    print(f'{name:<20} {dpd:>8.4f} {eod:>8.4f}')
print('\nDPD/EOD: lower = more fair | 0 = perfectly fair')

## 10. Fairness Visualization

In [ ]:
names = list(gf_res.keys()); x=np.arange(len(names)); w=0.35
fig,axes=plt.subplots(1,2,figsize=(15,6))

for ax,fr,title in [(axes[0],gf_res,'Gender Fairness'),(axes[1],af_res,'Age Fairness')]:
    dpds=[fr[n]['dpd'] for n in names]; eods=[fr[n]['eod'] for n in names]
    ax.bar(x-w/2, dpds, w, label='DPD', color='#2E75B6', edgecolor='white')
    ax.bar(x+w/2, eods, w, label='EOD', color='#C00000', edgecolor='white')
    ax.set_title(f'{title} Before vs After Compression', fontweight='bold', fontsize=12)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Fairness Gap (lower=fairer)'); ax.legend(); ax.grid(axis='y',alpha=0.3)

plt.suptitle('Fairness Impact of MLP Compression\nDPD=Demographic Parity Diff | EOD=Equalized Odds Diff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fairness_compression.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Full Comparison Table + Visualization

In [ ]:
print('='*90)
print('FULL D2 RESULTS — GERMAN CREDIT DATASET (mean ± std, 5 seeds)')
print('='*90)
print(f'\n{"Model":<30} {"Accuracy":>20} {"F1 Score":>20} {"AUC-ROC":>20}')
print('-'*90)

print('\n-- CLASSICAL ML (no PCA) --')
for name,r in ml_results.items():
    print(f'{name:<30} {np.mean(r["accuracy"]):.4f}±{np.std(r["accuracy"]):.4f}         {np.mean(r["f1"]):.4f}±{np.std(r["f1"]):.4f}         {np.mean(r["auc"]):.4f}±{np.std(r["auc"]):.4f}')

print('\n-- CLASSICAL ML (with PCA) --')
for name,r in ml_pca_results.items():
    print(f'{name}+PCA{"":<22} {np.mean(r["accuracy"]):.4f}±{np.std(r["accuracy"]):.4f}         {np.mean(r["f1"]):.4f}±{np.std(r["f1"]):.4f}         {np.mean(r["auc"]):.4f}±{np.std(r["auc"]):.4f}')

print('\n-- MLP VARIANTS --')
for label,r in [('MLP Base',mlp_res),('MLP Coreset',coreset_res),('MLP Quantized',quant_res)]:
    print(f'{label:<30} {np.mean(r["accuracy"]):.4f}±{np.std(r["accuracy"]):.4f}         {np.mean(r["f1"]):.4f}±{np.std(r["f1"]):.4f}         {np.mean(r["auc"]):.4f}±{np.std(r["auc"]):.4f}')
for s in SPARSITY:
    r=prun_res[s]
    print(f'{f"MLP Pruned {int(s*100)}%":<30} {np.mean(r["accuracy"]):.4f}±{np.std(r["accuracy"]):.4f}         {np.mean(r["f1"]):.4f}±{np.std(r["f1"]):.4f}         {np.mean(r["auc"]):.4f}±{np.std(r["auc"]):.4f}')

print('\n-- FAIRNESS SUMMARY --')
print(f'{"Scenario":<20} {"Gender DPD":>12} {"Gender EOD":>12} {"Age DPD":>10} {"Age EOD":>10}')
print('-'*66)
for n in names:
    print(f'{n:<20} {gf_res[n]["dpd"]:>12.4f} {gf_res[n]["eod"]:>12.4f} {af_res[n]["dpd"]:>10.4f} {af_res[n]["eod"]:>10.4f}')
print('\nLower DPD/EOD = more fair')

In [ ]:
# Bar chart — all models accuracy
all_n,all_m,all_s,all_c=[],[],[],[]
for name,r in ml_results.items():
    all_n.append(name); all_m.append(np.mean(r['accuracy'])); all_s.append(np.std(r['accuracy'])); all_c.append('#2E75B6')
all_n.append('MLP Base'); all_m.append(np.mean(mlp_res['accuracy'])); all_s.append(np.std(mlp_res['accuracy'])); all_c.append('#1A6B3A')
for s in SPARSITY:
    all_n.append(f'Pruned {int(s*100)}%'); all_m.append(np.mean(prun_res[s]['accuracy'])); all_s.append(np.std(prun_res[s]['accuracy'])); all_c.append('#C00000')
all_n.append('Quantized'); all_m.append(np.mean(quant_res['accuracy'])); all_s.append(np.std(quant_res['accuracy'])); all_c.append('#7030A0')
all_n.append('Coreset'); all_m.append(np.mean(coreset_res['accuracy'])); all_s.append(np.std(coreset_res['accuracy'])); all_c.append('#E36C09')

fig,ax=plt.subplots(figsize=(16,6))
bars=ax.bar(all_n,all_m,yerr=all_s,color=all_c,capsize=6,edgecolor='white',error_kw={'elinewidth':2,'ecolor':'black'})
ax.set_title('All Models — Accuracy (mean±std, 5 seeds)',fontsize=14,fontweight='bold')
ax.set_ylabel('Accuracy'); ax.set_ylim(0.5,1.05)
ax.set_xticklabels(all_n,rotation=35,ha='right',fontsize=10); ax.grid(axis='y',alpha=0.3)
for bar,v in zip(bars,all_m): ax.text(bar.get_x()+bar.get_width()/2,v+0.015,f'{v:.3f}',ha='center',fontsize=8,fontweight='bold')
ax.legend(handles=[mpatches.Patch(color=c,label=l) for c,l in [('#2E75B6','Classical ML'),('#1A6B3A','MLP Base'),('#C00000','Pruned'),('#7030A0','Quantized'),('#E36C09','Coreset')]],loc='lower right')
plt.tight_layout()
plt.savefig('all_models_comparison.png',dpi=150,bbox_inches='tight')
plt.show()